# Paper #3: The Perceptron (Rosenblatt, 1958)

## Implementation: From Rosenblatt's Perceptron to the Classic Learning Rule

This notebook implements the key ideas from Rosenblatt's 1958 paper:

1. **Rosenblatt's Architecture**: S-units → A-units → R-units with random connectivity
2. **The α-system Learning Rule**: Value updates through reinforcement
3. **Statistical Separability**: Demonstrating P_a, P_c, and the separability condition
4. **The Classic Perceptron Learning Rule**: The simplified, modern version
5. **Linear Separability**: What the perceptron can and cannot learn

**Prerequisites**: Paper #1 (McCulloch-Pitts neurons), basic linear algebra (vectors, dot products)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from typing import Tuple, List

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: Rosenblatt's Original Architecture

Rosenblatt's perceptron has three layers:
- **S-units (Sensory)**: Binary input retina
- **A-units (Association)**: Randomly connected to S-units, with threshold activation
- **R-units (Response)**: Output decisions, mutually exclusive

The key insight: connections from S → A are **random**, but the system can still learn
through changes in the values (weights) of the A → R connections.

In [ ]:
class RosenblattPerceptron:
    """Rosenblatt's original perceptron architecture.
    
    Implements the three-layer system from the 1958 paper:
    S-units (retina) -> A-units (association) -> R-units (responses)
    
    Parameters
    ----------
    n_sensory : int
        Number of S-units (retina size).
    n_association : int
        Number of A-units (N_A in the paper).
    n_responses : int
        Number of R-units (mutually exclusive responses).
    n_excitatory : int
        Number of excitatory connections per A-unit (x in the paper).
    n_inhibitory : int
        Number of inhibitory connections per A-unit (y in the paper).
    threshold : int
        Threshold for A-unit activation (theta in the paper).
    system_type : str
        'alpha', 'beta', or 'gamma' (see Table 1 in the paper).
    """
    
    def __init__(
        self,
        n_sensory: int = 100,
        n_association: int = 500,
        n_responses: int = 2,
        n_excitatory: int = 10,
        n_inhibitory: int = 3,
        threshold: int = 5,
        system_type: str = 'alpha'
    ):
        self.n_s = n_sensory
        self.n_a = n_association
        self.n_r = n_responses
        self.x = n_excitatory
        self.y = n_inhibitory
        self.theta = threshold
        self.system_type = system_type
        
        # Random S -> A connections (the paper's key feature)
        # Each A-unit gets x excitatory and y inhibitory connections
        # from randomly chosen S-units
        self.excitatory_connections = np.zeros((n_association, n_sensory))
        self.inhibitory_connections = np.zeros((n_association, n_sensory))
        
        for j in range(n_association):
            # Randomly select x excitatory origin points
            exc_indices = np.random.choice(n_sensory, size=n_excitatory, replace=False)
            self.excitatory_connections[j, exc_indices] = 1
            
            # Randomly select y inhibitory origin points (from remaining)
            remaining = np.setdiff1d(np.arange(n_sensory), exc_indices)
            if len(remaining) >= n_inhibitory:
                inh_indices = np.random.choice(remaining, size=n_inhibitory, replace=False)
                self.inhibitory_connections[j, inh_indices] = 1
        
        # Values of A-units for each response (the learnable parameters)
        # V[r, j] = value of A-unit j for response r
        self.values = np.zeros((n_responses, n_association))
        
        # Assign A-units to source-sets (random, ~equal partition)
        # omega = proportion of R-units each A-unit is connected to
        assignments = np.random.randint(0, n_responses, size=n_association)
        self.source_sets = np.zeros((n_responses, n_association), dtype=bool)
        for r in range(n_responses):
            self.source_sets[r] = (assignments == r)
    
    def compute_a_units(self, stimulus: np.ndarray) -> np.ndarray:
        """Compute A-unit activations (predominant phase).
        
        Each A-unit fires if:
        sum(excitatory inputs) - sum(inhibitory inputs) >= threshold
        """
        excitation = self.excitatory_connections @ stimulus  # (n_a,)
        inhibition = self.inhibitory_connections @ stimulus  # (n_a,)
        net_input = excitation - inhibition
        return (net_input >= self.theta).astype(float)
    
    def respond(self, stimulus: np.ndarray) -> int:
        """Compute the perceptron's response to a stimulus.
        
        The response with the greatest total impulse wins
        (postdominant phase — mean-discriminating / mu-system).
        """
        a_activations = self.compute_a_units(stimulus)
        
        # For each response, compute the total impulse from its source-set
        impulses = np.zeros(self.n_r)
        for r in range(self.n_r):
            # Active A-units in this source-set, weighted by their values
            active_in_set = a_activations * self.source_sets[r]
            impulses[r] = np.sum(active_in_set * self.values[r])
        
        return int(np.argmax(impulses))
    
    def reinforce(self, stimulus: np.ndarray, correct_response: int):
        """Apply reinforcement to update A-unit values.
        
        Implements the alpha-system rule from Table 1:
        Active A-units in the correct response's source-set gain +1.
        """
        a_activations = self.compute_a_units(stimulus)
        
        if self.system_type == 'alpha':
            # Alpha system: active units in source-set gain +1
            active_in_set = a_activations * self.source_sets[correct_response]
            self.values[correct_response] += active_in_set
            
        elif self.system_type == 'gamma':
            # Gamma system: active gain, inactive lose (parasitic)
            source = self.source_sets[correct_response]
            active_in_set = a_activations * source
            inactive_in_set = (1 - a_activations) * source
            n_active = np.sum(active_in_set)
            n_inactive = np.sum(inactive_in_set)
            
            self.values[correct_response] += active_in_set  # active gain +1
            if n_inactive > 0:
                # Inactive units lose proportionally
                loss = n_active / n_inactive if n_inactive > 0 else 0
                self.values[correct_response] -= inactive_in_set * loss
    
    def train(
        self,
        stimuli: np.ndarray,
        labels: np.ndarray,
        n_epochs: int = 10
    ) -> List[float]:
        """Train the perceptron on a set of stimuli with forced responses."""
        accuracies = []
        
        for epoch in range(n_epochs):
            # Shuffle training order
            indices = np.random.permutation(len(stimuli))
            correct = 0
            
            for idx in indices:
                stimulus = stimuli[idx]
                label = labels[idx]
                
                # Test response
                response = self.respond(stimulus)
                if response == label:
                    correct += 1
                
                # Reinforce correct response (forced learning)
                self.reinforce(stimulus, label)
            
            accuracies.append(correct / len(stimuli))
        
        return accuracies

### Demo: Learning to Distinguish Two Stimulus Classes

Let's create a "differentiated environment" — two classes of stimuli
that activate different regions of the retina — and watch the perceptron learn.

In [ ]:
def generate_stimuli(
    n_sensory: int,
    n_per_class: int,
    class_regions: List[Tuple[int, int]],
    activation_prob: float = 0.3,
    noise_prob: float = 0.05
) -> Tuple[np.ndarray, np.ndarray]:
    """Generate stimuli from two classes with different retinal activation patterns.
    
    Class 0: activates primarily in class_regions[0]
    Class 1: activates primarily in class_regions[1]
    """
    stimuli = []
    labels = []
    
    for class_idx, (start, end) in enumerate(class_regions):
        for _ in range(n_per_class):
            stimulus = np.zeros(n_sensory)
            # Activate S-units in the class region with high probability
            region_size = end - start
            stimulus[start:end] = (np.random.rand(region_size) < activation_prob).astype(float)
            # Add noise outside the region
            stimulus[:start] = (np.random.rand(start) < noise_prob).astype(float)
            stimulus[end:] = (np.random.rand(n_sensory - end) < noise_prob).astype(float)
            
            stimuli.append(stimulus)
            labels.append(class_idx)
    
    return np.array(stimuli), np.array(labels)


# Create the perceptron and stimuli
N_SENSORY = 100
N_ASSOCIATION = 500

# Class 0: activates left half of retina (S-units 0-49)
# Class 1: activates right half of retina (S-units 50-99)
train_stimuli, train_labels = generate_stimuli(
    n_sensory=N_SENSORY,
    n_per_class=50,
    class_regions=[(0, 50), (50, 100)],
    activation_prob=0.3,
    noise_prob=0.05
)

# Test with novel stimuli (generalization!)
test_stimuli, test_labels = generate_stimuli(
    n_sensory=N_SENSORY,
    n_per_class=25,
    class_regions=[(0, 50), (50, 100)],
    activation_prob=0.3,
    noise_prob=0.05
)

print(f"Training set: {len(train_stimuli)} stimuli ({sum(train_labels==0)} class 0, {sum(train_labels==1)} class 1)")
print(f"Test set: {len(test_stimuli)} stimuli ({sum(test_labels==0)} class 0, {sum(test_labels==1)} class 1)")
print(f"\nPerceptron: {N_SENSORY} S-units, {N_ASSOCIATION} A-units, 2 R-units")

In [ ]:
# Train the Rosenblatt perceptron
perceptron = RosenblattPerceptron(
    n_sensory=N_SENSORY,
    n_association=N_ASSOCIATION,
    n_responses=2,
    n_excitatory=10,
    n_inhibitory=3,
    threshold=5,
    system_type='alpha'
)

# Track both recall (P_r) and generalization (P_g)
n_epochs = 30
train_accuracies = []
test_accuracies = []

for epoch in range(n_epochs):
    # Measure before training this epoch
    train_correct = sum(perceptron.respond(s) == l for s, l in zip(train_stimuli, train_labels))
    test_correct = sum(perceptron.respond(s) == l for s, l in zip(test_stimuli, test_labels))
    
    train_accuracies.append(train_correct / len(train_stimuli))
    test_accuracies.append(test_correct / len(test_stimuli))
    
    # Train one epoch
    indices = np.random.permutation(len(train_stimuli))
    for idx in indices:
        perceptron.reinforce(train_stimuli[idx], train_labels[idx])

# Plot learning curves (analogous to Figures 7-8 in the paper)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(n_epochs), train_accuracies, 'b-o', label='$P_r$ (Recall — training stimuli)', markersize=5)
ax.plot(range(n_epochs), test_accuracies, 'r-s', label='$P_g$ (Generalization — novel stimuli)', markersize=5)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax.set_xlabel('Epoch (number of passes through training set)')
ax.set_ylabel('Probability of correct response')
ax.set_title('Rosenblatt Perceptron: Learning Curves\n(α-system, differentiated environment)')
ax.legend()
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal P_r (recall): {train_accuracies[-1]:.3f}")
print(f"Final P_g (generalization): {test_accuracies[-1]:.3f}")
print(f"\nKey paper prediction: In a differentiated environment,")
print(f"P_g should approach P_r as training increases. ✓" if abs(train_accuracies[-1] - test_accuracies[-1]) < 0.15 else "")

### Effect of System Size (N_A)

Rosenblatt showed (Figures 7-8) that performance improves dramatically
with the number of association cells. Let's verify this.

In [ ]:
# Test different N_A values
n_a_values = [50, 100, 200, 500, 1000]
results = {}

for n_a in n_a_values:
    p = RosenblattPerceptron(
        n_sensory=N_SENSORY,
        n_association=n_a,
        n_responses=2,
        n_excitatory=10,
        n_inhibitory=3,
        threshold=5,
        system_type='alpha'
    )
    
    accs = []
    for epoch in range(20):
        test_correct = sum(p.respond(s) == l for s, l in zip(test_stimuli, test_labels))
        accs.append(test_correct / len(test_stimuli))
        
        indices = np.random.permutation(len(train_stimuli))
        for idx in indices:
            p.reinforce(train_stimuli[idx], train_labels[idx])
    
    results[n_a] = accs

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
for n_a, accs in results.items():
    ax.plot(range(20), accs, '-o', label=f'$N_A$ = {n_a}', markersize=4)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax.set_xlabel('Epoch')
ax.set_ylabel('$P_g$ (Generalization accuracy)')
ax.set_title('Effect of Number of Association Cells ($N_A$) on Learning\n(Cf. Figures 7-8 in the paper)')
ax.legend()
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Paper prediction: More A-units → faster learning and higher asymptote ✓")

## Part 2: Statistical Separability — The Core Theory

Rosenblatt's central insight: learning works when the **statistical separability condition** is met:

$$P_{c12} < P_a < P_{c11}$$

Where:
- $P_a$ = proportion of A-units activated by a stimulus
- $P_{c11}$ = probability an A-unit responds to two stimuli from the **same** class
- $P_{c12}$ = probability an A-unit responds to stimuli from **different** classes

Let's measure these quantities directly.

In [ ]:
def measure_statistical_separability(
    perceptron: RosenblattPerceptron,
    stimuli: np.ndarray,
    labels: np.ndarray,
    n_pairs: int = 200
) -> dict:
    """Measure P_a, P_c11 (within-class), and P_c12 (between-class)."""
    # Measure P_a: average proportion of A-units activated
    p_a_values = []
    for s in stimuli:
        a = perceptron.compute_a_units(s)
        p_a_values.append(np.mean(a))
    p_a = np.mean(p_a_values)
    
    # Separate by class
    class_0 = stimuli[labels == 0]
    class_1 = stimuli[labels == 1]
    
    # Measure P_c11: within-class co-activation
    p_c11_values = []
    for _ in range(n_pairs):
        # Pick two random stimuli from the same class
        cls = class_0 if np.random.rand() < 0.5 else class_1
        i, j = np.random.choice(len(cls), size=2, replace=False)
        a_i = perceptron.compute_a_units(cls[i])
        a_j = perceptron.compute_a_units(cls[j])
        # P_c = P(a_j=1 | a_i=1)
        active_i = a_i > 0
        if np.sum(active_i) > 0:
            p_c11_values.append(np.mean(a_j[active_i]))
    p_c11 = np.mean(p_c11_values) if p_c11_values else 0
    
    # Measure P_c12: between-class co-activation
    p_c12_values = []
    for _ in range(n_pairs):
        i = np.random.randint(len(class_0))
        j = np.random.randint(len(class_1))
        a_i = perceptron.compute_a_units(class_0[i])
        a_j = perceptron.compute_a_units(class_1[j])
        active_i = a_i > 0
        if np.sum(active_i) > 0:
            p_c12_values.append(np.mean(a_j[active_i]))
    p_c12 = np.mean(p_c12_values) if p_c12_values else 0
    
    return {'P_a': p_a, 'P_c11': p_c11, 'P_c12': p_c12}


# Measure for our trained perceptron
stats = measure_statistical_separability(perceptron, train_stimuli, train_labels)

print("Statistical Separability Analysis")
print("=" * 45)
print(f"P_a   (activation probability):     {stats['P_a']:.4f}")
print(f"P_c11 (within-class co-activation):  {stats['P_c11']:.4f}")
print(f"P_c12 (between-class co-activation): {stats['P_c12']:.4f}")
print(f"\nSeparability condition: P_c12 < P_a < P_c11")
print(f"  {stats['P_c12']:.4f} < {stats['P_a']:.4f} < {stats['P_c11']:.4f}")

condition_met = stats['P_c12'] < stats['P_a'] < stats['P_c11']
print(f"  Condition met: {'✓ YES — learning should succeed!' if condition_met else '✗ NO — learning may fail'}")

## Part 3: The Classic Perceptron Learning Rule

Rosenblatt's paper described the full biological architecture. Over time,
the perceptron was simplified to the form most commonly taught today:

**A single neuron that learns a linear decision boundary.**

The modern Perceptron Learning Rule:
1. Compute output: $\hat{y} = \text{sign}(\mathbf{w} \cdot \mathbf{x} + b)$
2. If correct: do nothing
3. If wrong: update weights:
   - $\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot y \cdot \mathbf{x}$
   - $b \leftarrow b + \eta \cdot y$

This is the **bivalent system** from the paper: positive reinforcement for correct,
negative for incorrect.

In [ ]:
class Perceptron:
    """Classic Perceptron with the learning rule.
    
    This is the simplified, modern version of Rosenblatt's perceptron.
    A single threshold unit that learns a linear decision boundary.
    
    Parameters
    ----------
    n_features : int
        Number of input features.
    learning_rate : float
        Step size for weight updates (eta).
    """
    
    def __init__(self, n_features: int, learning_rate: float = 1.0):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
    
    def predict(self, x: np.ndarray) -> int:
        """Predict class label (+1 or -1) for a single input."""
        activation = np.dot(self.weights, x) + self.bias
        return 1 if activation >= 0 else -1
    
    def train(
        self,
        X: np.ndarray,
        y: np.ndarray,
        n_epochs: int = 100
    ) -> List[dict]:
        """Train the perceptron using the Perceptron Learning Rule.
        
        Returns history of errors and weights per epoch.
        """
        history = []
        
        for epoch in range(n_epochs):
            errors = 0
            
            for xi, yi in zip(X, y):
                prediction = self.predict(xi)
                
                if prediction != yi:
                    # Update rule: w <- w + eta * y * x
                    self.weights += self.lr * yi * xi
                    self.bias += self.lr * yi
                    errors += 1
            
            history.append({
                'epoch': epoch,
                'errors': errors,
                'weights': self.weights.copy(),
                'bias': self.bias
            })
            
            # Convergence: no errors
            if errors == 0:
                break
        
        return history

### Demo: Learning AND, OR (linearly separable)

The perceptron can learn any **linearly separable** function.
AND and OR are linearly separable — the perceptron should converge.

In [ ]:
# Define AND and OR datasets (labels: +1 / -1)
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

y_and = np.array([-1, -1, -1, 1])   # AND: only (1,1) -> +1
y_or = np.array([-1, 1, 1, 1])       # OR: only (0,0) -> -1

# Train AND perceptron
p_and = Perceptron(n_features=2)
history_and = p_and.train(X_logic, y_and, n_epochs=20)

# Train OR perceptron
p_or = Perceptron(n_features=2)
history_or = p_or.train(X_logic, y_or, n_epochs=20)

# Verify
print("AND Gate:")
print(f"  Converged in {len(history_and)} epochs")
print(f"  Weights: {p_and.weights}, Bias: {p_and.bias}")
for x in X_logic:
    print(f"  {x} -> {p_and.predict(x):+d}")

print(f"\nOR Gate:")
print(f"  Converged in {len(history_or)} epochs")
print(f"  Weights: {p_or.weights}, Bias: {p_or.bias}")
for x in X_logic:
    print(f"  {x} -> {p_or.predict(x):+d}")

### Visualizing the Decision Boundary

The perceptron learns a **linear decision boundary** (a line in 2D, a hyperplane in higher dimensions).

The boundary is where $\mathbf{w} \cdot \mathbf{x} + b = 0$.

In [ ]:
def plot_decision_boundary(perceptron: Perceptron, X: np.ndarray, y: np.ndarray, title: str):
    """Plot the perceptron's decision boundary in 2D."""
    # Create a mesh grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predict for each point in the grid
    Z = np.array([perceptron.predict(np.array([x, y_val])) 
                  for x, y_val in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)
    
    # Plot
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    plt.scatter(X[y == -1][:, 0], X[y == -1][:, 1], c='red', s=200, 
               edgecolors='black', label='Class -1', zorder=5)
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], c='blue', s=200, 
               edgecolors='black', label='Class +1', zorder=5)
    
    # Draw the decision boundary line
    w = perceptron.weights
    b = perceptron.bias
    if abs(w[1]) > 1e-10:
        x_line = np.linspace(x_min, x_max, 100)
        y_line = -(w[0] * x_line + b) / w[1]
        plt.plot(x_line, y_line, 'k-', linewidth=2, label='Decision boundary')
    
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.title(title)
    plt.legend()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plt.sca(axes[0])
plot_decision_boundary(p_and, X_logic, y_and, 'AND Gate — Linearly Separable ✓')

plt.sca(axes[1])
plot_decision_boundary(p_or, X_logic, y_or, 'OR Gate — Linearly Separable ✓')

plt.tight_layout()
plt.show()

## Part 4: The XOR Problem — What the Perceptron Cannot Learn

Rosenblatt himself noted (p. 405) that "statistical separability alone does not provide a
sufficient basis for higher order abstraction." The most famous example is **XOR**.

XOR is **not linearly separable** — no single line can separate the +1 and -1 points.
This limitation will be rigorously proved by Minsky & Papert (Paper #4, 1969).

In [ ]:
# XOR: the perceptron's nemesis
y_xor = np.array([-1, 1, 1, -1])  # XOR: (0,0)->-1, (0,1)->+1, (1,0)->+1, (1,1)->-1

p_xor = Perceptron(n_features=2)
history_xor = p_xor.train(X_logic, y_xor, n_epochs=100)

# Plot the error curve — it never reaches zero!
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Error curve
axes[0].plot([h['errors'] for h in history_xor], 'r-o', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Number of errors')
axes[0].set_title('XOR: Error never reaches zero!\n(Perceptron oscillates indefinitely)')
axes[0].axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Target: 0 errors')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decision boundary attempt
plt.sca(axes[1])
plot_decision_boundary(p_xor, X_logic, y_xor, 'XOR — NOT Linearly Separable ✗')

plt.tight_layout()
plt.show()

print("XOR Results:")
print(f"  After {len(history_xor)} epochs, errors = {history_xor[-1]['errors']}")
print(f"  The perceptron CANNOT learn XOR!")
print(f"\n  Why? No single line can separate (0,0),(1,1) from (0,1),(1,0).")
print(f"  This will be formally proved by Minsky & Papert (1969) — Paper #4.")
print(f"  The solution: MULTI-LAYER networks — Paper #6 (Backpropagation, 1986).")

## Part 5: A More Realistic Example — Classifying Patterns

Let's use the classic perceptron to learn a practical classification task:
distinguishing between two types of 2D patterns (like Rosenblatt's
square vs. circle discrimination, but in a simplified feature space).

In [ ]:
# Generate two linearly separable clusters
np.random.seed(42)
n_samples = 100

# Class +1: centered at (2, 2)
X_pos = np.random.randn(n_samples, 2) * 0.8 + np.array([2, 2])
# Class -1: centered at (-2, -2)
X_neg = np.random.randn(n_samples, 2) * 0.8 + np.array([-2, -2])

X = np.vstack([X_pos, X_neg])
y = np.array([1] * n_samples + [-1] * n_samples)

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X, y = X[shuffle_idx], y[shuffle_idx]

# Split into train/test
X_train, y_train = X[:150], y[:150]
X_test, y_test = X[150:], y[150:]

# Train
p_cluster = Perceptron(n_features=2, learning_rate=0.1)
history = p_cluster.train(X_train, y_train, n_epochs=50)

# Evaluate
test_predictions = np.array([p_cluster.predict(x) for x in X_test])
test_accuracy = np.mean(test_predictions == y_test)

print(f"Converged in {len(history)} epochs")
print(f"Training errors in final epoch: {history[-1]['errors']}")
print(f"Test accuracy: {test_accuracy:.1%}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Learning curve
axes[0].plot([h['errors'] for h in history], 'b-o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Number of misclassifications')
axes[0].set_title('Perceptron Learning Curve\n(Two Gaussian clusters)')
axes[0].grid(True, alpha=0.3)

# Decision boundary
plt.sca(axes[1])
plot_decision_boundary(p_cluster, X_test, y_test, 
                       f'Learned Decision Boundary\n(Test accuracy: {test_accuracy:.1%})')

plt.tight_layout()
plt.show()

## Part 6: The Perceptron Convergence Theorem

One of the most important theoretical results associated with the perceptron:

> **If the training data is linearly separable, the perceptron learning rule
> is guaranteed to converge in a finite number of steps.**

This was proved formally by Novikoff (1962), building on Rosenblatt's work.
Let's verify this empirically by running multiple trials.

In [ ]:
def generate_linearly_separable(n_samples: int, margin: float = 1.0) -> Tuple[np.ndarray, np.ndarray]:
    """Generate linearly separable data with a given margin."""
    X = np.random.randn(n_samples, 2)
    # True boundary: x1 + x2 = 0
    # Add margin: only keep points with |x1 + x2| > margin
    distances = X[:, 0] + X[:, 1]
    keep = np.abs(distances) > margin
    X = X[keep]
    y = np.sign(X[:, 0] + X[:, 1]).astype(int)
    return X, y


# Run convergence experiment
margins = [0.5, 1.0, 2.0, 3.0]
n_trials = 20

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, margin in enumerate(margins):
    ax = axes[idx // 2][idx % 2]
    
    convergence_epochs = []
    for trial in range(n_trials):
        X_sep, y_sep = generate_linearly_separable(200, margin=margin)
        p = Perceptron(n_features=2)
        hist = p.train(X_sep, y_sep, n_epochs=500)
        convergence_epochs.append(len(hist))
    
    ax.hist(convergence_epochs, bins=15, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Epochs to convergence')
    ax.set_ylabel('Count')
    ax.set_title(f'Margin = {margin}\n(mean: {np.mean(convergence_epochs):.1f} epochs)')
    ax.grid(True, alpha=0.3)

fig.suptitle('Perceptron Convergence Theorem: Always converges for linearly separable data\n'
             'Larger margin → faster convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight: The perceptron ALWAYS converges when data is linearly separable.")
print("Larger margin between classes → faster convergence.")
print("This is the Perceptron Convergence Theorem (Novikoff, 1962).")

## Part 7: Weight Evolution — Watching the Perceptron Learn

Let's visualize how the weight vector evolves during training.
The weight vector is perpendicular to the decision boundary — as it
rotates, the boundary moves until it correctly separates the classes.

In [ ]:
# Generate simple data
np.random.seed(123)
X_vis, y_vis = generate_linearly_separable(100, margin=1.5)

# Train and record weight history
p_vis = Perceptron(n_features=2)
history_vis = p_vis.train(X_vis, y_vis, n_epochs=50)

# Select key epochs to show
n_epochs_total = len(history_vis)
if n_epochs_total >= 6:
    show_epochs = [0, 1, 2, n_epochs_total // 3, 2 * n_epochs_total // 3, n_epochs_total - 1]
else:
    show_epochs = list(range(min(6, n_epochs_total)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for plot_idx, epoch_idx in enumerate(show_epochs[:6]):
    ax = axes[plot_idx // 3][plot_idx % 3]
    h = history_vis[epoch_idx]
    
    # Create a temporary perceptron with the weights at this epoch
    p_temp = Perceptron(n_features=2)
    p_temp.weights = h['weights']
    p_temp.bias = h['bias']
    
    # Plot
    plt.sca(ax)
    plot_decision_boundary(p_temp, X_vis, y_vis, 
                          f"Epoch {h['epoch']} — {h['errors']} errors")

fig.suptitle('Perceptron Learning: Weight Evolution Over Time\n'
             'Watch the decision boundary rotate until it separates the classes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Converged in {n_epochs_total} epochs — the boundary found the correct orientation!")

## Summary: From Rosenblatt to Modern Neural Networks

### What Rosenblatt Established (1958)

| Concept | Rosenblatt's Version | Modern Form |
|---|---|---|
| **Architecture** | S → A → R (three layers, random S→A) | Input → Hidden → Output |
| **Learning** | Reinforce active A-units for correct response | Gradient descent on loss function |
| **Decision rule** | Response with greatest total impulse wins | argmax of output layer |
| **Convergence** | Proved for statistical separability | Perceptron Convergence Theorem |

### The Fundamental Limitation

The perceptron can only learn **linearly separable** functions:
- ✓ AND, OR, NAND, NOR
- ✗ XOR, XNOR
- ✗ Any function requiring a curved or non-linear boundary

### What Comes Next

- **Paper #4 (Minsky & Papert, 1969)**: Formal proof of these limitations
- **Paper #6 (Rumelhart, Hinton, Williams, 1986)**: Backpropagation — training
  *multi-layer* perceptrons that CAN learn XOR and any other function

Rosenblatt's perceptron is simple, but it established the **fundamental paradigm**:
a machine that learns from data by adjusting connection strengths.
Every neural network trained today is a descendant of this idea.